# Лабораторная работа №5 - Ансамбли моделей машинного обучения. Часть 1.

### Цель лабораторной работы: изучение ансамблей моделей машинного обучения.

### Задание:
1. Выберите набор данных (датасет) для решения задачи классификации или регресии.
2. В случае необходимости проведите удаление или заполнение пропусков и кодирование категориальных признаков.
3. С использованием метода train_test_split разделите выборку на обучающую и тестовую.
4. Обучите следующие ансамблевые модели: две модели группы бэггинга (бэггинг или случайный лес или сверхслучайные деревья); AdaBoost; градиентный бустинг.
5. Оцените качество моделей с помощью одной из подходящих для задачи метрик. Сравните качество полученных моделей.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import (BaggingClassifier, RandomForestClassifier, 
                               AdaBoostClassifier, GradientBoostingClassifier)
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

In [2]:
df = pd.read_csv('loan_approval_dataset.csv')
print("Размер датасета:", df.shape)
print("\nПервые 5 строк:")
df.head()

Размер датасета: (4269, 13)

Первые 5 строк:


,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected


In [3]:
print("Пропуски в данных:")
print(df.isnull().sum())

Пропуски в данных:
loan_id                      0
 no_of_dependents            0
 education                   0
 self_employed               0
 income_annum                0
 loan_amount                 0
 loan_term                   0
 cibil_score                 0
 residential_assets_value    0
 commercial_assets_value     0
 luxury_assets_value         0
 bank_asset_value            0
 loan_status                 0
dtype: int64


In [4]:
# Удаляем loan_id
df = df.drop(columns=['loan_id'])

# Убираем пробелы в названиях столбцов
df.columns = df.columns.str.strip()

# Кодируем текстовые столбцы в цифры
le = LabelEncoder()
df['education'] = le.fit_transform(df['education'])        # Graduate=0, Not Graduate=1
df['self_employed'] = le.fit_transform(df['self_employed']) # No=0, Yes=1
df['loan_status'] = le.fit_transform(df['loan_status'])     # Approved=0, Rejected=1

df.head()

,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,2,0,0,9600000,29900000,12,778,2400000,17600000,22700000,8000000,0
1,0,1,1,4100000,12200000,8,417,2700000,2200000,8800000,3300000,1
2,3,0,0,9100000,29700000,20,506,7100000,4500000,33300000,12800000,1
3,3,0,0,8200000,30700000,8,467,18200000,3300000,23300000,7900000,1
4,5,1,1,9800000,24200000,20,382,12400000,8200000,29400000,5000000,1


In [5]:
# X - все столбцы кроме loan_status (признаки)
# y - только loan_status (то что предсказываем)
X = df.drop(columns=['loan_status'])
y = df['loan_status']

# Делим: 80% обучение, 20% тест
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Обучающая выборка:", X_train.shape)
print("Тестовая выборка:", X_test.shape)

Обучающая выборка: (3415, 11)
Тестовая выборка: (854, 11)


In [7]:
# Создаём все 4 модели
model_bagging = BaggingClassifier(n_estimators=100, random_state=42)
model_forest = RandomForestClassifier(n_estimators=100, random_state=42)
model_ada = AdaBoostClassifier(n_estimators=100, random_state=42)
model_gb = GradientBoostingClassifier(n_estimators=100, random_state=42)

# Обучаем все на обучающей выборке
model_bagging.fit(X_train, y_train)
model_forest.fit(X_train, y_train)
model_ada.fit(X_train, y_train)
model_gb.fit(X_train, y_train)

print("Все модели обучены")

Все модели обучены


In [8]:
# Проверяем каждую модель на тестовых данных
models = {
    'Bagging': model_bagging,
    'Random Forest': model_forest,
    'AdaBoost': model_ada,
    'Gradient Boosting': model_gb
}

for name, model in models.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    print(f"{name}: точность = {acc:.4f} ({acc*100:.2f}%)")

Bagging: точность = 0.9824 (98.24%)
Random Forest: точность = 0.9778 (97.78%)
AdaBoost: точность = 0.9672 (96.72%)
Gradient Boosting: точность = 0.9778 (97.78%)


In [11]:
print("=" * 45)
print("  СРАВНЕНИЕ АНСАМБЛЕВЫХ МОДЕЛЕЙ")
print("=" * 45)

results = {}
for name, model in models.items():
    y_pred = model.predict(X_test)
    results[name] = accuracy_score(y_test, y_pred)

for name, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    bar = "█" * int(acc * 20)
    print(f"{name:<20} {acc*100:.2f}%  {bar}")

print("=" * 45)
best = max(results, key=results.get)
print(f"  Лучшая модель: {best}")
print("=" * 45)

  СРАВНЕНИЕ АНСАМБЛЕВЫХ МОДЕЛЕЙ
Bagging              98.24%  ███████████████████
Random Forest        97.78%  ███████████████████
Gradient Boosting    97.78%  ███████████████████
AdaBoost             96.72%  ███████████████████
  Лучшая модель: Bagging
